In [ ]:
import pandas as pd
import numpy as np
import optuna

import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

## Import des données

In [ ]:
df = pd.read_csv("../../data/resample_normalized_flagged_v3.csv")

## Préparation des données

In [ ]:
df = df.drop(columns=["apogee_id", "STARFLAGS"])

class_columns = ['class_spectral', 'class_lum_logg', 'class_lum_jhk', 'class_lum_bins_logg', 'class_lum_bins_jhk']

chemical_columns = ['C_FE', 'CI_FE', 'N_FE', 'O_FE', 'NA_FE', 'MG_FE', 'AL_FE', 'SI_FE', 'S_FE', 'K_FE', 'CA_FE', 'TI_FE', 'V_FE', 'CR_FE', 'MN_FE', 'NI_FE', 'FE_H']
physique_columns = ['J', 'H', 'K', 'LOGG', 'M_H', 'VMICRO', 'VMACRO']

df_chem = df.drop(columns=physique_columns)
df_phys = df.drop(columns=chemical_columns)

In [ ]:
target_column = 'class_lum_bins_jhk'
X = df_chem.drop(columns=class_columns)
y = df_chem[target_column]

In [ ]:
# Calcul des fréquences des classes
frequencies = df_chem[target_column].value_counts(normalize=True)

# Calcul des poids inverses
weights = (1 / frequencies).to_dict()

### Implémentation de LightGBM

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_LGB, X_test_LGB, y_train_LGB, y_test_LGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, class_weight=weights, random_state=42, objective="multiclass", num_class=3)

lgb_model.fit(X_train_LGB, y_train_LGB, eval_set=[(X_test_LGB, y_test_LGB)])

y_pred_LGB = lgb_model.predict(X_test_LGB)

In [ ]:
#évaluation du modèle
print("Accuracy : \n", accuracy_score(y_test_LGB, y_pred_LGB))
print("Classification Report : \n", classification_report(y_test_LGB, y_pred_LGB))

In [ ]:
f1_LGB = f1_score(y_test_LGB, y_pred_LGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_LGB)

In [ ]:
y_probs_LGB = lgb_model.predict_proba(X_test_LGB)  # Probabilités prédites des classes

auc_pr_list_LGB = []
for idx, class_name in enumerate(np.unique(y_test_LGB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_LGB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_LGB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_LGB.append(auc_pr)
auc_pr_LGB = np.mean(auc_pr_list_LGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_LGB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_LGB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

### Implémentation de HistGradientBoosting

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_HGB, X_test_HGB, y_train_HGB, y_test_HGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sample_weights = compute_sample_weight(class_weight=weights, y=y_train_HGB)

# Initialisation du modèle
hgb = HistGradientBoostingClassifier(loss="log_loss", learning_rate=0.1, max_iter=100)

# Entraînement
hgb.fit(X_train_HGB, y_train_HGB, sample_weight=sample_weights)

# Prédiction
y_pred_HGB = hgb.predict(X_test_HGB)

In [ ]:
# Évaluation
print("Accuracy:", accuracy_score(y_test_HGB, y_pred_HGB))
print("Classification Report:\n", classification_report(y_test_HGB, y_pred_HGB))

In [ ]:
f1_HGB = f1_score(y_test_HGB, y_pred_HGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_HGB)

In [ ]:
y_probs_HGB = hgb.predict_proba(X_test_HGB)  # Probabilités prédites des classes

auc_pr_list_HGB = []
for idx, class_name in enumerate(np.unique(y_test_HGB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_HGB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_HGB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_HGB.append(auc_pr)
auc_pr_HGB = np.mean(auc_pr_list_HGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_HGB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_HGB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

### Implémentation de CatBoost

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_CB, X_test_CB, y_train_CB, y_test_CB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Création et entraînement du modèle
model_catboost = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, loss_function='MultiClass',  eval_metric='MultiClass', class_weights=weights, verbose=100)

model_catboost.fit(X_train_CB, y_train_CB, eval_set=(X_test_CB, y_test_CB), early_stopping_rounds=100)

# Prédictions
y_pred = model_catboost.predict(X_test_CB)

In [ ]:
# Performance
print("Accuracy:", accuracy_score(y_test_CB, y_pred))
print(classification_report(y_test_CB, y_pred))

In [ ]:
f1_CB = f1_score(y_test_CB, y_pred, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_CB)

In [ ]:
y_probs_CB = model_catboost.predict_proba(X_test_CB)  # Probabilités prédites des classes

auc_pr_list_CB = []
for idx, class_name in enumerate(np.unique(y_test_CB)):
    # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
    y_true_binary = (y_test_CB == class_name).astype(int)

    # Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_CB[:, idx])

    # Calculer l'AUC-PR
    auc_pr = auc(recall, precision)

    # Ajouter à la liste
    auc_pr_list_CB.append(auc_pr)
auc_pr_CB = np.mean(auc_pr_list_CB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_CB)

In [ ]:
# Tracé de la courbe PR
plt.figure(figsize=(6, 6))
plt.plot(recall, precision, marker='o', label=f'AUC-PR = {auc_pr_CB:.2f}')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.grid()
plt.show()

## Optimisation avec optuna

### opti LightGBM

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_LGB, X_test_LGB, y_train_LGB, y_test_LGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective_LGB(trial):
    params = {
        'objective': 'multiclass',
        'metric': 'multi_logloss',
        'num_class': 3,
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 16),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
    }

    lgb_model = lgb.LGBMClassifier(**params, class_weight=weights, random_state=42)
    lgb_model.fit(X_train_LGB, 
              y_train_LGB, 
              eval_set=[(X_test_LGB, y_test_LGB)], 
              )

    y_pred = lgb_model.predict(X_test_LGB)
    y_probs_LGB = lgb_model.predict_proba(X_test_LGB)

    f1 = f1_score(y_test_LGB, y_pred, average='weighted')
    auc_pr_list_LGB = []
    for idx, class_name in enumerate(np.unique(y_test_LGB)):
        # Créer une version binaire de y_test_LGB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_LGB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_LGB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_LGB.append(auc_pr)
    auc_pr_LGB = np.mean(auc_pr_list_LGB)  # Moyenne sur toutes les classes


    return f1 , auc_pr_LGB

# Créer un objet d'étude Optuna
study_LGB = optuna.create_study(directions=['maximize', 'maximize'])
# Optimiser la fonction objectif
study_LGB.optimize(objective_LGB, n_trials=100, show_progress_bar=True)

In [ ]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_LGB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_LGB.best_trials[0].params)
print("Meilleur score F1 : ", study_LGB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_LGB.best_trials[0].values[1])

### opti HistGradientBoosting

In [ ]:
# Division des données
X_train_HGB, X_test_HGB, y_train_HGB, y_test_HGB = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Poids des classes
sample_weights_HGB = compute_sample_weight(class_weight="balanced", y=y_train_HGB)

def objective_HGB(trial):
    """ Fonction d'optimisation pour Optuna """

    # Hyperparamètres à optimiser
    params = {
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        "max_iter": trial.suggest_int("max_iter", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_loguniform("l2_regularization", 1e-4, 1e-1)
    }

    # Initialisation du modèle
    model_HGB = HistGradientBoostingClassifier(loss="log_loss", **params, random_state=42)

    # Entraînement avec les poids
    model_HGB.fit(X_train_HGB, y_train_HGB, sample_weight=sample_weights_HGB)

    # Prédiction
    y_pred_HGB = model_HGB.predict(X_test_HGB)
    y_probs_HGB = model_HGB.predict_proba(X_test_HGB)

    # Calcul du F1-score pondéré (mieux adapté au déséquilibre)
    score = f1_score(y_test_HGB, y_pred_HGB, average="weighted")
    # Calcul l'AUC-PR
    auc_pr_list_HGB = []
    for idx, class_name in enumerate(np.unique(y_test_HGB)):
        # Créer une version binaire de y_test_HGB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_HGB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_HGB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_HGB.append(auc_pr)
    auc_pr_HGB = np.mean(auc_pr_list_HGB)  # Moyenne sur toutes les classes

    return score, auc_pr_HGB  # Optuna va maximiser ces valeurs

# Lancer l'optimisation
study_HGB = optuna.create_study(directions=['maximize', 'maximize'])
study_HGB.optimize(objective_HGB, n_trials=50, show_progress_bar=True)

In [ ]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_HGB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_HGB.best_trials[0].params)
print("Meilleur score F1 : ", study_HGB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_HGB.best_trials[0].values[1])

### opti CatBoost

In [ ]:
# Division des données
X_train_CB, X_test_CB, y_train_CB, y_test_CB = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Poids des classes pour gérer le déséquilibre
sample_weights_CB = compute_sample_weight(class_weight="balanced", y=y_train_CB)

def objective_CB(trial):
    """ Fonction d'optimisation pour Optuna """

    # Hyperparamètres à optimiser
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000, step=250),
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        "depth": trial.suggest_int("depth", 4, 12),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_uniform("bagging_temperature", 0, 1),
    }

    # Création et entraînement du modèle
    model_CB = CatBoostClassifier(
        **params, 
        loss_function="MultiClass", 
        eval_metric="MultiClass", 
        verbose=0, 
        random_seed=42
    )

    model_CB.fit(X_train_CB, y_train_CB, sample_weight=sample_weights_CB, eval_set=(X_test_CB, y_test_CB), early_stopping_rounds=100, verbose=False)


    # Prédiction
    y_pred_CB = model_CB.predict(X_test_CB)
    y_probs_CB = model_CB.predict_proba(X_test_CB)

    # Calcul du F1-score pondéré (mieux adapté au déséquilibre)
    score = f1_score(y_test_CB, y_pred_CB, average="weighted")
    # Calcul l'AUC-PR
    auc_pr_list_CB = []
    for idx, class_name in enumerate(np.unique(y_test_CB)):
        # Créer une version binaire de y_test_CB : 1 pour la classe courante, 0 pour les autres
        y_true_binary = (y_test_CB == class_name).astype(int)
        
        # Précision-Rappel
        precision, recall, _ = precision_recall_curve(y_true_binary, y_probs_CB[:, idx])
        
        # Calculer l'AUC-PR
        auc_pr = auc(recall, precision)
        
        # Ajouter à la liste
        auc_pr_list_CB.append(auc_pr)
    auc_pr_CB = np.mean(auc_pr_list_CB)  # Moyenne sur toutes les classes

    return score, auc_pr_CB  # Optuna va maximiser ces valeurs

# Lancer l'optimisation
study_CB = optuna.create_study(directions=['maximize', 'maximize'])
study_CB.optimize(objective_CB, n_trials=50, show_progress_bar=True)

In [ ]:
# Afficher les meilleurs hyperparamètres trouvés
print("Meilleur essai : ", study_CB.best_trials[0])
print("Meilleurs hyperparamètres : ", study_CB.best_trials[0].params)
print("Meilleur score F1 : ", study_CB.best_trials[0].values[0])
print("Meilleur score AUC-PR : ", study_CB.best_trials[0].values[1])